# Astral Audio
Generates a daily playlist based on your astrological chart for the day

**Pipeline:**
- Compute natal + transit chart from birth info and filter active aspects by orb limits
- Call Gemini to select the most significant aspects and generate a horoscope
- Build audio feature target vector from selected aspects
- Load music library for scoring (user upload recommended for model training - export Liked Songs from [exportify.net](https://exportify.net) and set path in config)
- Refine target vector using personal Lasso model trained on user uploaded playlist
- Score and rank tracks against the blended target vector

## 1 - Imports & Config

In [ ]:
%pip install kerykeion google-genai scikit-learn requests --quiet

In [2]:
# --- File Paths ---
# paths are relative to the repo root
BASE_PATH          = os.path.abspath('')
LOCAL_LIBRARY_PATH = os.path.join(BASE_PATH, 'music_library.csv')
USER_PLAYLIST_PATH = None  # set to path of Exportify CSV - defaults to local library
MODEL_PATH         = os.path.join(BASE_PATH, 'personal_model.pkl')  # saved locally, not in repo

# --- Gemini API Keys ---
# set API key and BYPASS_GEMINI=0 to generate personal chart - defaults to stub info
os.environ['GEMINI_API_KEY'] = ''
os.environ['BYPASS_GEMINI'] = '1'

# --- Birth Info ---
# stub info to bypass API call - replace with your own details to generate a personal chart
birth_info = {
    'date': '2000-01-01',
    'time': '00:00',
    'lat':  40.7128,
    'lng':  -74.0060,
    'tz':   'America/New_York'
    }

# set to None to default to current time
transit_dt = '2026-05-31 00:00'

# set to current location
transit_loc = {
    'lat': 34.0522,
    'lng': -118.2437,
    'tz':  'America/Los_Angeles'
    }

## 2 - Natal & Transit Aspects

- Compute natal chart from birth data and transit chart from current datetime and location
- Filter to five aspect types (conjunction, opposition, trine, square, sextile) within moiety orb limits

In [ ]:
daily_aspects, natal_chart, transit_chart = get_transit_aspects(birth_info, transit_dt=transit_dt, transit_loc=transit_loc)

print(f'Active aspects ({len(daily_aspects)}):')
print(format_aspects(daily_aspects))

## 3 - LLM Horoscope

- Pass all filtered aspects to Gemini with curated system prompt 
- Returns daily horoscope and keywords + 3 most significant aspects with interpretations

In [ ]:
horoscope = get_horoscope(daily_aspects)

print('Daily Summary:')
print(horoscope['daily_summary'])
print()
print('Keywords:', ', '.join(horoscope['daily_keywords']))
print()
print('Aspects:')
for a in horoscope['aspects']:
    print(f"\n{a['aspect']}")
    print(f"{a['meaning']}")

## 4 - Baseline Target Vector

- Planets have manually defined audio profiles (valence, energy, danceability, acousticness, tempo, mode) based on astrological character
- LLM-selected aspects are averaged into a single target vector, weighted by signal strength (tighter orb = stronger signal)
- Target vector represents baseline astrological audio profile before ML refinement

In [ ]:
select_aspects = get_select_aspects(daily_aspects, horoscope)
baseline_vector = build_target_vector(select_aspects)

print('Select Aspects:')
for a in select_aspects:
    print(f'  Natal {a.p1_name} in {a.aspect} with transiting {a.p2_name} '
          f'(orb: {abs(a.orbit):.2f}°)')

print()
print('Baseline Target Vector:')
for feature, value in baseline_vector.items():
    if feature == 'tempo':
        print(f'  {feature:<16}: {value:.1f} BPM')
    elif feature == 'mode':
        label = {1: 'major', 0: 'minor', None: 'no preference'}.get(value)
        print(f'  {feature:<16}: {label}')
    else:
        print(f'  {feature:<16}: {value:.3f}')

## 5 - Music Library

Loads full music library, supports genre/decade filtering in-app
- user upload - merges new tracks and trains personal model
- local library only - uses baseline target vector

In [ ]:
library = load_music_library(local_library_path = LOCAL_LIBRARY_PATH)

print(f'Library loaded: {len(library)} tracks')

## 6 - Personal Lasso Model

Refine baseline target vector with user taste profile:
- training signal - song save date used as a weak proxy for mood
- feature engineering - active aspects computed for each save date, signal strengths encoded as sparse vector
- target - deviation of each song's audio features from user's personal mean, anchors model to user's baseline music taste and targets mood shifts
- model output - one LassoCV per audio feature

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)

# stub mode - skip model, use baseline vector only
if os.environ.get('BYPASS_GEMINI') == '1':
    model_bundle = None
    print('Stub mode — skipping model training.')

# train if user upload exists + has save dates or load existing model
elif USER_PLAYLIST_PATH and Path(USER_PLAYLIST_PATH).exists():
    try:
        raw_df = pd.read_csv(USER_PLAYLIST_PATH)
        raw_df.columns = [c.lower().replace(' ', '_').replace('(s)', 's') for c in raw_df.columns]
        if 'added_at' in raw_df.columns:
            model_bundle = train_model(raw_df, birth_info)
            save_model(model_bundle, MODEL_PATH)
            print('Model trained and saved.')
        else:
            model_bundle = load_model(MODEL_PATH) if Path(MODEL_PATH).exists() else None
            print('No added_at column — loaded existing model.' if model_bundle else 'No added_at column and no saved model.')
    except Exception as e:
        model_bundle = load_model(MODEL_PATH) if Path(MODEL_PATH).exists() else None
        print(f'Training failed ({e}) — loaded existing model.' if model_bundle else f'Training failed ({e}) — falling back to baseline vector.')

elif Path(MODEL_PATH).exists():
    model_bundle = load_model(MODEL_PATH)
    print('Loaded existing model.')

else:
    model_bundle = None
    print('No liked songs upload and no saved model — using baseline vector.')

if model_bundle:
    print()
    print('User baseline:')
    for f, v in model_bundle['user_mean'].items():
        print(f'  {f:<16}: {v:.3f}')

## 7 - Score Tracks & Output Playlist

Tracks scored by weighted Euclidean distance from target vector (lower score = closer match):
- baseline target - no model exists or pipeline is in stub mode
- blended target - 30% model prediction + 70% baseline


In [ ]:
# blend model prediction with baseline vector if model is available
if model_bundle is not None and daily_aspects:
    model_vector = predict_target_vector(daily_aspects, model_bundle)
    target_vector = blend_target_vectors(model_vector, baseline_vector, model_weight=0.3)
    print('Target vector: blended (30% model, 70% baseline)')
else:
    target_vector = baseline_vector
    print('Target vector: baseline only')

# print target vector values
print()
for feature, value in target_vector.items():
    if feature == 'tempo':
        print(f'  {feature:<16}: {value:.1f} BPM')
    elif feature == 'mode':
        label = {1: 'major', 0: 'minor', None: 'no preference'}.get(value)
        print(f'  {feature:<16}: {label}')
    else:
        print(f'  {feature:<16}: {value:.3f}')

print()
playlist = score_tracks(library, target_vector)

print('=' * 55)
print('YOUR ASTRAL AUDIO PLAYLIST')
print('=' * 55)
print()
print(horoscope['daily_summary'])
print()
print(f"Today's energy: {', '.join(horoscope['daily_keywords'])}")
print()
for i, row in playlist.iterrows():
    print(f"{i+1:>2}. {row['track_name']} — {row['artist_names']}  (score: {row['score']:.3f})")